# Notebook 13b — EN vs DE Input Comparison: Jaccard Similarity

**Core question:** Does giving LLMs German input improve agreement with pipeline NER
systems compared to English translations?

## Experimental design

| Condition | Input text | Entity output language | Text limit |
|---|---|---|---|
| A — EN | `content_en` (English translation) | English | 8000 chars |
| B — DE | `content` (German original) | German | 8000 chars |
| Pipelines | `content` (German original) | German | Full article |

**Why 8000 chars:** At 3000 chars LLMs could reach only 52% of pipeline entities by
position. At 8000 chars coverage is 96.6%. All runs in this notebook use 8000 chars.

**Models:** Qwen 2.5 7B · Llama 3.1 8B · Qwen3 32B · Llama 3.3 70B

**DE prompt note:** LLMs instructed to return entity text *exactly as it appears in the
source* — prevents silent translation of German entity names to English.

**Qwen 7B provider:** Runs via HuggingFace Inference API (tries multiple providers).
If all are exhausted the run skips cleanly and continues with Groq models.


In [ ]:
# ── CELL 1 : INSTALLATION ────────────────────────────────────────────────────
!pip install -q groq huggingface_hub
!pip install -q 'numpy>=2.0'   # must be last


In [ ]:
# ── CELL 2 : IMPORTS & CONFIGURATION ─────────────────────────────────────────

import os, json, pickle, time, re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

ROOT        = Path('/content/drive/MyDrive/thesis')
DATA_PROC   = ROOT / 'Project/Data/Processed'
FIGURES_DIR = ROOT / 'Project/Outputs/Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

GROQ_TOKEN = userdata.get('GROQ_TOKEN')
HF_TOKEN   = userdata.get('HF_TOKEN')

# ── Text limit ────────────────────────────────────────────────────────────────
TEXT_LIMIT = 8000   # covers 96.6% of entities by position

# ── Checkpointing ─────────────────────────────────────────────────────────────
SAVE_EVERY               = 10
MAX_CONSECUTIVE_FAILURES = 5
MAX_FAILURE_RATE         = 0.20
GROQ_SLEEP               = 2.0
HF_SLEEP                 = 3.0
RETRY_DELAYS             = [4, 8, 16]

# ── Models ────────────────────────────────────────────────────────────────────
# Each model runs in both EN and DE conditions.
# provider: 'groq' or 'hf'
MODELS = {
    "Qwen/Qwen2.5-7B-Instruct"    : {"scale_B": 7,  "family": "qwen25", "provider": "hf"},
    "llama-3.1-8b-instant"        : {"scale_B": 8,  "family": "llama",  "provider": "groq"},
    "qwen/qwen3-32b"              : {"scale_B": 32, "family": "qwen3",  "provider": "groq"},
    "llama-3.3-70b-versatile"     : {"scale_B": 70, "family": "llama",  "provider": "groq"},
}
CONDITIONS = ["en", "de"]

# ── HF providers to try for Qwen 7B (in order) ───────────────────────────────
HF_PROVIDERS = ["together", "novita", "sambanova", "nebius", "fireworks-ai"]

# ── Pipeline entity columns ───────────────────────────────────────────────────
PIPELINE_ENTITY_COLS = {
    "spaCy"  : "ner_spacy",
    "Stanza" : "ner_stanza",
    "Flair"  : "ner_flair",
}

# ── NER prompts ───────────────────────────────────────────────────────────────
NER_SYSTEM_PROMPT_EN = """You are a named entity recognition (NER) system.
Extract ALL named entities from the text provided.

Return ONLY a valid JSON object — no markdown, no explanation, no preamble:
{
  "entities": [
    {"text": "<entity surface form>", "label": "<TYPE>"}
  ]
}

Use exactly these four labels:
  PER   -- person names
  LOC   -- locations, countries, cities, geographical features
  ORG   -- organisations, companies, institutions
  MISC  -- other named entities (events, products, languages, nationalities, etc.)

If no entities are found return:  {"entities": []}"""

NER_SYSTEM_PROMPT_DE = """You are a named entity recognition (NER) system.
Extract ALL named entities from the German text provided.
CRITICAL: Return entity text EXACTLY as it appears in the German source — do NOT translate to English.
For example: use "Deutschland" not "Germany", "München" not "Munich", "Bundesregierung" not "Federal Government".

Return ONLY a valid JSON object — no markdown, no explanation, no preamble:
{
  "entities": [
    {"text": "<entity text exactly as in source>", "label": "<TYPE>"}
  ]
}

Use exactly these four labels:
  PER   -- person names
  LOC   -- locations, countries, cities, geographical features
  ORG   -- organisations, companies, institutions
  MISC  -- other named entities (events, products, languages, nationalities, etc.)

If no entities are found return:  {"entities": []}"""

NER_USER_TEMPLATE = "Extract named entities from the following text:\n\n{text}"

print("✓ Configuration loaded")
print(f"  Groq token : {'SET' if GROQ_TOKEN else '⚠ MISSING'}")
print(f"  HF token   : {'SET' if HF_TOKEN else '⚠ MISSING (Qwen 7B will be skipped)'}")
print(f"  Text limit : {TEXT_LIMIT} chars")
print(f"  Models     : {list(MODELS.keys())}")
print(f"  Conditions : {CONDITIONS}")
print(f"  Total runs : {len(MODELS) * len(CONDITIONS)}")


In [ ]:
# ── CELL 3 : CHECK AVAILABLE GROQ MODELS ─────────────────────────────────────

from groq import Groq
groq_client = Groq(api_key=GROQ_TOKEN)

print("Querying Groq...")
try:
    available_groq = {m.id for m in groq_client.models.list().data}
    print(f"  {len(available_groq)} models available")
except Exception as e:
    available_groq = set()
    print(f"  ⚠ Could not list: {e}")

for model_id, meta in MODELS.items():
    if meta["provider"] != "groq":
        continue
    status = "✓" if model_id in available_groq else "✗ NOT FOUND"
    print(f"  {status} {model_id} ({meta['scale_B']}B)")

print("\nQwen 7B → HF Inference API (provider availability checked at runtime)")


In [ ]:
# ── CELL 4 : LOAD EXISTING DATA ──────────────────────────────────────────────

print("Loading pipeline results...")
pipeline_df = pd.read_pickle(DATA_PROC / 'ner_pipeline_results.pkl')
print(f"  shape   : {pipeline_df.shape}")

# Verify columns
for name, col in PIPELINE_ENTITY_COLS.items():
    ok = "✓" if col in pipeline_df.columns else "⚠ MISSING"
    print(f"  {ok} {name}: '{col}'")
for col in ['content', 'content_en', 'article_id']:
    ok = "✓" if col in pipeline_df.columns else "⚠ MISSING"
    print(f"  {ok} column '{col}'")

# ── Article IDs — from Qwen 7B checkpoint (183 articles, same as all prev work) ──
qwen7b_ckpt_path = DATA_PROC / 'ner_llm_checkpoint.pkl'
if qwen7b_ckpt_path.exists():
    with open(qwen7b_ckpt_path, 'rb') as f:
        qwen7b_old = pickle.load(f)
    # Old format: flat dict {article_id: {status, entities}}
    ARTICLE_IDS_183 = [
        aid for aid, data in qwen7b_old.items()
        if isinstance(data, dict) and data.get('status') != 'api_error'
    ]
    print(f"\n  Working set: {len(ARTICLE_IDS_183)} articles (from Qwen7B checkpoint)")
else:
    ARTICLE_IDS_183 = list(pipeline_df['article_id'].unique())
    print(f"\n  Working set: {len(ARTICLE_IDS_183)} articles (from pipeline_df)")

# ── Text lookups ──────────────────────────────────────────────────────────────
idx = pipeline_df.set_index('article_id')
id_to_de = idx['content'].to_dict()       # German original
id_to_en = idx['content_en'].to_dict()    # English translation

print(f"  ✓ German text lookup   : {len(id_to_de)} entries")
print(f"  ✓ English text lookup  : {len(id_to_en)} entries")


In [ ]:
# ── CELL 5 : CHECKPOINT UTILITIES ────────────────────────────────────────────

def ckpt_path(model_id: str, condition: str) -> Path:
    """Returns checkpoint path for a given model + condition."""
    safe = re.sub(r'[/:\\.]', '_', model_id)
    return DATA_PROC / f'nb13b_{safe}_{condition}_8k_checkpoint.pkl'

def load_checkpoint(model_id: str, condition: str) -> dict:
    fp = ckpt_path(model_id, condition)
    if fp.exists():
        with open(fp, 'rb') as f:
            ckpt = pickle.load(f)
        done = len([r for r in ckpt['results'] if r['status'] == 'ok'])
        fail = len(ckpt['failed_ids'])
        print(f"  ↺ Resuming {model_id} [{condition}]: {done} done, {fail} failed")
        return ckpt
    return {
        'model_id'   : model_id,
        'condition'  : condition,
        'text_limit' : TEXT_LIMIT,
        'results'    : [],
        'failed_ids' : [],
        'stop_reason': None,
        'started_at' : datetime.now().isoformat(),
        'updated_at' : None,
    }

def save_checkpoint(ckpt: dict, model_id: str, condition: str):
    ckpt['updated_at'] = datetime.now().isoformat()
    with open(ckpt_path(model_id, condition), 'wb') as f:
        pickle.dump(ckpt, f)

def is_complete(ckpt: dict) -> bool:
    return ckpt.get('stop_reason') == 'complete'

def processed_ids(ckpt: dict) -> set:
    return {r['article_id'] for r in ckpt['results']} | set(ckpt['failed_ids'])

print("✓ Checkpoint utilities ready")
print("  Naming: nb13b_{model}_{condition}_8k_checkpoint.pkl")


In [ ]:
# ── CELL 6 : EXTRACTION FUNCTIONS ────────────────────────────────────────────

def _parse_entities(raw: str) -> list:
    """
    Parse JSON entity list. Handles:
    - Qwen3 think blocks (closed and truncated)
    - Markdown fences
    - Empty strings
    - Truncated JSON
    """
    raw = raw.strip()
    if not raw:
        return []
    if '<think>' in raw:
        if '</think>' in raw:
            raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
        else:
            brace_pos = raw.rfind('{')
            raw = raw[brace_pos:] if brace_pos != -1 else ''
    raw = re.sub(r'^```(?:json)?\s*', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'\s*```$',          '', raw, flags=re.MULTILINE)
    raw = raw.strip()
    try:
        data = json.loads(raw)
        return [e for e in data.get('entities', [])
                if isinstance(e, dict) and 'text' in e and 'label' in e]
    except json.JSONDecodeError:
        return []


def build_messages(model_id: str, text: str, condition: str) -> tuple:
    """
    Returns (messages, max_tokens) tuned per model family and condition.
    condition: 'en' or 'de'
    """
    system = NER_SYSTEM_PROMPT_EN if condition == 'en' else NER_SYSTEM_PROMPT_DE
    user   = NER_USER_TEMPLATE.format(text=text[:TEXT_LIMIT])

    if 'qwen3' in model_id.lower():
        # Qwen3: suppress thinking, extra token budget
        messages   = [{"role": "system", "content": system},
                      {"role": "user",   "content": "/no_think\n\n" + user}]
        max_tokens = 2048

    else:
        # Standard (Llama, Qwen 7B)
        messages   = [{"role": "system", "content": system},
                      {"role": "user",   "content": user}]
        max_tokens = 1024   # increased from 512 — 8k input may yield more entities

    return messages, max_tokens


def extract_ner_groq(client, model_id: str, text: str, condition: str):
    """
    Groq NER call. Returns (entity_list, raw_str).
    Raises RuntimeError on unrecoverable failure.
    """
    from groq import RateLimitError, APITimeoutError, APIStatusError

    messages, max_tokens = build_messages(model_id, text, condition)
    last_exc = None

    for attempt, delay in enumerate([0] + RETRY_DELAYS, start=1):
        if delay:
            print(f"    Waiting {delay}s (retry {attempt})...")
            time.sleep(delay)
        try:
            resp = client.chat.completions.create(
                model=model_id, messages=messages,
                max_tokens=max_tokens, temperature=0.0,
            )
            raw = resp.choices[0].message.content or ''
            return _parse_entities(raw), raw
        except RateLimitError as e:
            last_exc = e
            if attempt <= len(RETRY_DELAYS):
                print(f"    429 rate-limit (attempt {attempt}), backing off...")
            else:
                raise RuntimeError(f"RATE_LIMIT: {e}") from e
        except APITimeoutError as e:
            last_exc = e
            if attempt <= len(RETRY_DELAYS):
                print(f"    Timeout (attempt {attempt}), retrying...")
            else:
                raise RuntimeError(f"TIMEOUT: {e}") from e
        except APIStatusError as e:
            raise RuntimeError(f"API_STATUS_{e.status_code}: {e.message}") from e
        except Exception as e:
            raise RuntimeError(f"UNEXPECTED: {e}") from e
    raise RuntimeError(f"Exhausted retries: {last_exc}")


def extract_ner_hf(model_id: str, text: str, condition: str):
    """
    HuggingFace Inference API NER call.
    Tries each provider in HF_PROVIDERS in sequence.
    Returns (entity_list, raw_str) or raises RuntimeError.
    """
    from huggingface_hub import InferenceClient

    messages, max_tokens = build_messages(model_id, text, condition)
    provider_errors = {}

    for provider in HF_PROVIDERS:
        try:
            client = InferenceClient(provider=provider, api_key=HF_TOKEN)
            resp   = client.chat.completions.create(
                model=model_id, messages=messages,
                max_tokens=max_tokens, temperature=0.0,
            )
            raw = resp.choices[0].message.content or ''
            return _parse_entities(raw), raw
        except Exception as e:
            err_str = str(e)
            provider_errors[provider] = err_str[:80]
            # 402 = credits exhausted on this provider → try next
            if '402' in err_str or 'credit' in err_str.lower() or 'quota' in err_str.lower():
                continue
            # Other errors (model not supported, timeout) → try next
            continue

    # All providers failed
    raise RuntimeError(f"ALL_HF_PROVIDERS_EXHAUSTED: {provider_errors}")


print("✓ Extraction functions ready")
print("  _parse_entities : think blocks · truncation · fences · empty")
print("  build_messages  : Qwen3 (no_think+2048) · others (std+1024)")
print("  extract_ner_groq: Llama 8B, Qwen3 32B, Llama 70B")
print("  extract_ner_hf  : Qwen 7B (tries all HF providers)")


In [ ]:
# ── CELL 7 : RUN LOOP ────────────────────────────────────────────────────────
# Re-running is always safe — resumes from checkpoint automatically.
# Stops and saves on: rate_limit / consecutive_failures / high_failure_rate.

def run_model(model_id, condition, article_ids, id_to_text,
              extract_fn, sleep_between):
    """
    Generic run loop for any model + condition.
    extract_fn signature: (text, condition) -> (entities, raw)
    """
    ckpt = load_checkpoint(model_id, condition)

    if is_complete(ckpt):
        n_ok = len([r for r in ckpt['results'] if r['status'] == 'ok'])
        print(f"  ✓ {model_id} [{condition.upper()}] already complete ({n_ok} articles)")
        return ckpt

    already_done = processed_ids(ckpt)
    remaining    = [aid for aid in article_ids if aid not in already_done]

    if not remaining:
        ckpt['stop_reason'] = 'complete'
        save_checkpoint(ckpt, model_id, condition)
        print(f"  ✓ {model_id} [{condition.upper()}] — all done, marked complete")
        return ckpt

    print(f"\n{'='*64}")
    print(f"  MODEL     : {model_id}")
    print(f"  CONDITION : {condition.upper()} input — {'English translation' if condition=='en' else 'German original'}")
    print(f"  TEXT LIMIT: {TEXT_LIMIT} chars")
    print(f"  Pending   : {len(remaining)} articles  (done: {len(already_done)})")
    print(f"{'='*64}")

    consec_fail = 0
    total_ok    = len([r for r in ckpt['results'] if r['status'] == 'ok'])
    total_fail  = len(ckpt['failed_ids'])

    for i, article_id in enumerate(remaining, start=1):
        text = id_to_text.get(article_id, '')
        if not text:
            ckpt['failed_ids'].append(article_id)
            consec_fail += 1; total_fail += 1
            print(f"  [{i}/{len(remaining)}] {article_id} — NO TEXT")
        else:
            try:
                entities, raw = extract_fn(text, condition)
                ckpt['results'].append({
                    'article_id'   : article_id,
                    'entities'     : entities,
                    'raw_response' : raw,
                    'status'       : 'ok',
                })
                consec_fail  = 0; total_ok += 1
                if i % 20 == 0 or i == len(remaining):
                    print(f"  [{i}/{len(remaining)}]  ok={total_ok}  "
                          f"fail={total_fail}  last_ents={len(entities)}")
                time.sleep(sleep_between)

            except RuntimeError as exc:
                err_msg = str(exc)
                consec_fail += 1; total_fail += 1
                ckpt['failed_ids'].append(article_id)
                print(f"\n  ⚠ [{i}/{len(remaining)}] FAILED [{article_id}]: {err_msg[:120]}")

                stop_reason = None
                if 'RATE_LIMIT' in err_msg or 'API_STATUS_402' in err_msg:
                    stop_reason = 'rate_limit'
                    print(f"\n  ✋ STOPPING — rate limit. Re-run to resume.")
                elif 'ALL_HF_PROVIDERS_EXHAUSTED' in err_msg:
                    stop_reason = 'hf_exhausted'
                    print(f"\n  ✋ STOPPING — all HF providers exhausted for Qwen 7B.")
                    print(f"     Continuing with Groq models is unaffected.")
                elif consec_fail >= MAX_CONSECUTIVE_FAILURES:
                    stop_reason = 'consecutive_failures'
                    print(f"\n  ✋ STOPPING — {consec_fail} consecutive failures.")
                else:
                    total_seen = total_ok + total_fail
                    if total_seen >= 20 and (total_fail/total_seen) > MAX_FAILURE_RATE:
                        stop_reason = 'high_failure_rate'
                        print(f"\n  ✋ STOPPING — failure rate "
                              f"{total_fail/total_seen*100:.1f}% > threshold.")

                if stop_reason:
                    ckpt['stop_reason'] = stop_reason
                    save_checkpoint(ckpt, model_id, condition)
                    print(f"     Checkpoint: {ckpt_path(model_id, condition).name}")
                    print(f"     Progress  : {total_ok} ok / {total_fail} failed / "
                          f"{len(remaining)-i} remaining")
                    return ckpt

        if i % SAVE_EVERY == 0:
            save_checkpoint(ckpt, model_id, condition)
            print(f"  💾 saved at article {i}")

    ckpt['stop_reason'] = 'complete'
    save_checkpoint(ckpt, model_id, condition)
    print(f"\n  ✅ COMPLETE: {model_id} [{condition.upper()}]  "
          f"ok={total_ok}  failed={total_fail}")
    return ckpt

print("✓ Run loop ready")


In [ ]:
# ── CELL 8 : EXECUTE ALL RUNS ────────────────────────────────────────────────
# 8 total runs: 4 models × 2 conditions (EN + DE).
# Re-running this cell is safe — resumes from checkpoints.
# Qwen 7B skipped automatically if all HF providers are exhausted.

all_results = {}   # {(model_id, condition): checkpoint_dict}

for model_id, meta in MODELS.items():
    for condition in CONDITIONS:
        run_key = (model_id, condition)

        # Choose text source and extract function
        id_to_text  = id_to_en if condition == 'en' else id_to_de
        sleep_time  = HF_SLEEP if meta['provider'] == 'hf' else GROQ_SLEEP

        if meta['provider'] == 'hf':
            if not HF_TOKEN:
                print(f"\n  – Skipping {model_id} [{condition.upper()}] — HF_TOKEN not set")
                continue
            extract_fn = lambda text, cond, mid=model_id: extract_ner_hf(mid, text, cond)
        else:
            extract_fn = lambda text, cond, mid=model_id: extract_ner_groq(
                groq_client, mid, text, cond)

        ckpt = run_model(
            model_id      = model_id,
            condition     = condition,
            article_ids   = ARTICLE_IDS_183,
            id_to_text    = id_to_text,
            extract_fn    = extract_fn,
            sleep_between = sleep_time,
        )
        all_results[run_key] = ckpt

# ── Run summary ───────────────────────────────────────────────────────────────
print("\n" + "="*64)
print("RUN SUMMARY")
print("="*64)
for (mid, cond), ckpt in all_results.items():
    n_ok  = len([r for r in ckpt['results'] if r['status'] == 'ok'])
    n_fail= len(ckpt['failed_ids'])
    print(f"  {mid:<40} [{cond.upper()}]  "
          f"ok={n_ok:3d}  fail={n_fail:2d}  [{ckpt.get('stop_reason','?')}]")


In [ ]:
# ── CELL 9 : CONSOLIDATE ALL RESULTS ─────────────────────────────────────────

LABEL_NORM = {
    "PERSON":"PER","person":"PER",
    "LOCATION":"LOC","location":"LOC","GPE":"LOC","gpe":"LOC","FAC":"LOC",
    "ORGANIZATION":"ORG","organisation":"ORG","organization":"ORG",
    "MISCELLANEOUS":"MISC","miscellaneous":"MISC",
    "EVENT":"MISC","PRODUCT":"MISC","LANGUAGE":"MISC","NORP":"MISC",
    "WORK_OF_ART":"MISC","LAW":"MISC","DATE":"MISC","TIME":"MISC","POL":"MISC",
    "PER":"PER","LOC":"LOC","ORG":"ORG","MISC":"MISC",
}
VALID_LABELS = {"PER","LOC","ORG","MISC"}

def normalize_entities(entity_list) -> set:
    out = set()
    if not entity_list:
        return out
    for e in entity_list:
        if isinstance(e, dict):
            text  = str(e.get('text','')).strip().lower()
            label = LABEL_NORM.get(str(e.get('label','')), None)
        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            text  = str(e[0]).strip().lower()
            label = LABEL_NORM.get(str(e[1]), None)
        else:
            continue
        if text and label in VALID_LABELS:
            out.add((text, label))
    return out

def build_entity_lookup(ckpt) -> dict:
    """Handles both OLD flat dict and NEW results list checkpoint formats."""
    if isinstance(ckpt, dict) and 'results' in ckpt:
        return {r['article_id']: normalize_entities(r.get('entities',[]))
                for r in ckpt['results'] if r.get('status') == 'ok'}
    else:
        return {aid: normalize_entities(data.get('entities',[]))
                for aid, data in ckpt.items()
                if isinstance(data,dict) and data.get('status') != 'api_error'}

# ── Build LLM entity lookups per (model, condition) ───────────────────────────
llm_lookups = {}   # {(model_id, condition): {article_id: set_of_(text,label)}}

for (model_id, condition), ckpt in all_results.items():
    n_ok = len([r for r in ckpt['results'] if r.get('status') == 'ok'])
    if n_ok > 0:
        llm_lookups[(model_id, condition)] = build_entity_lookup(ckpt)
        print(f"  ✓ {model_id} [{condition.upper()}]: {n_ok} articles")
    else:
        print(f"  ✗ {model_id} [{condition.upper()}]: 0 ok — excluded from Jaccard")

# ── Pipeline entity lookups ───────────────────────────────────────────────────
pipeline_lookups = {}
article_id_set   = set(ARTICLE_IDS_183)

for name, col in PIPELINE_ENTITY_COLS.items():
    if col not in pipeline_df.columns:
        print(f"  ⚠ Skipping {name}: '{col}' missing"); continue
    lookup = {}
    for _, row in pipeline_df.iterrows():
        aid = row['article_id']
        if aid not in article_id_set: continue
        raw = row[col]
        lookup[aid] = normalize_entities(raw) if isinstance(raw, (list,tuple)) else set()
    pipeline_lookups[name] = lookup
    print(f"  ✓ Pipeline {name}: {len(lookup)} articles")

print("\n✓ All lookups ready")


In [ ]:
# ── CELL 10 : COMPUTE JACCARD SIMILARITY ──────────────────────────────────────

def jaccard(a: set, b: set) -> float:
    if not a and not b: return 1.0
    if not a or  not b: return 0.0
    return len(a & b) / len(a | b)

def jaccard_stats(p_lookup, l_lookup, article_ids) -> dict:
    scores = [jaccard(p_lookup.get(aid,set()), l_lookup.get(aid,set()))
              for aid in article_ids if aid in p_lookup and aid in l_lookup]
    return {'mean'  : float(np.mean(scores))   if scores else float('nan'),
            'std'   : float(np.std(scores))    if scores else float('nan'),
            'median': float(np.median(scores)) if scores else float('nan'),
            'n'     : len(scores)}

# Build model metadata lookup (scale_B, family)
model_meta = {mid: meta for mid, meta in MODELS.items()}

# Compute Jaccard for every pipeline × LLM × condition combination
records = []
for (model_id, condition), l_lookup in llm_lookups.items():
    for pipeline_name, p_lookup in pipeline_lookups.items():
        stats = jaccard_stats(p_lookup, l_lookup, ARTICLE_IDS_183)
        records.append({
            'pipeline'  : pipeline_name,
            'model'     : model_id,
            'condition' : condition,
            'scale_B'   : model_meta.get(model_id, {}).get('scale_B', 0),
            'family'    : model_meta.get(model_id, {}).get('family', '?'),
            **stats,
        })

jdf = pd.DataFrame(records)

# ── Print matrix: Flair (highest pipeline) across conditions ──────────────────
print("Jaccard matrix — Flair pipeline (mean):")
flair_df = jdf[jdf['pipeline'] == 'Flair'].copy()
pivot = flair_df.pivot_table(
    index='model', columns='condition', values='mean', aggfunc='first'
).round(3)
pivot = pivot.reindex(sorted(pivot.index,
    key=lambda m: model_meta.get(m,{}).get('scale_B',0)))
if 'en' in pivot.columns and 'de' in pivot.columns:
    pivot['delta (DE-EN)'] = (pivot['de'] - pivot['en']).round(3)
print(pivot.to_string())

print("\nJaccard matrix — all pipelines (mean, EN condition):")
en_df = jdf[jdf['condition'] == 'en'].copy()
pivot2 = en_df.pivot_table(
    index='pipeline', columns='model', values='mean', aggfunc='first'
).round(3)
print(pivot2.to_string())

jdf.to_csv(DATA_PROC / 'nb13b_jaccard_results.csv', index=False)
print("\n✓ Saved → nb13b_jaccard_results.csv")


In [ ]:
# ── CELL 11 : PLOTS ──────────────────────────────────────────────────────────
# Figure 1 : Scaling curve — EN condition (Condition A)
# Figure 2 : Scaling curve — DE condition (Condition B)
# Figure 3 : EN vs DE delta per model (main finding)

PIPELINE_STYLES = {
    'Flair' : {'color': '#1f77b4', 'marker': 'o', 'lw': 2.2, 'ms': 9},
    'Stanza': {'color': '#ff7f0e', 'marker': 's', 'lw': 2.2, 'ms': 9},
    'spaCy' : {'color': '#2ca02c', 'marker': '^', 'lw': 2.2, 'ms': 9},
}
CONDITION_STYLE = {
    'en': {'ls': '--', 'alpha': 0.85, 'label_suffix': ' (EN input)'},
    'de': {'ls': '-',  'alpha': 1.00, 'label_suffix': ' (DE input)'},
}

def scale_xlabel(model_id, scale_B):
    s = int(scale_B)
    if 'qwen3'  in model_id.lower(): return f"Qwen3\n{s}B"
    if 'Qwen2'  in model_id or 'qwen2' in model_id.lower(): return f"Qwen2.5\n{s}B"
    if '8b'     in model_id.lower(): return f"Llama\n{s}B"
    if '70b'    in model_id.lower(): return f"Llama\n{s}B"
    return f"{s}B"

model_order = sorted(model_meta.keys(),
                     key=lambda m: model_meta[m].get('scale_B', 0))

# ── Figure 1 & 2 : Scaling curves per condition ───────────────────────────────
for cond, cond_label in [('en','Condition A — EN Input'), ('de','Condition B — DE Input')]:
    sub = jdf[jdf['condition'] == cond]
    if sub.empty:
        print(f"No data for condition {cond} — skipping plot"); continue

    fig, ax = plt.subplots(figsize=(10, 6))
    for pname, style in PIPELINE_STYLES.items():
        psub = sub[sub['pipeline'] == pname].sort_values('scale_B')
        if psub.empty: continue
        ax.plot(psub['scale_B'], psub['mean'],
                label=pname, color=style['color'],
                marker=style['marker'], linewidth=style['lw'], markersize=style['ms'])
        ax.fill_between(psub['scale_B'],
                        psub['mean']-psub['std'], psub['mean']+psub['std'],
                        alpha=0.1, color=style['color'])

    scale_rows = sub.drop_duplicates('scale_B').sort_values('scale_B')
    ax.set_xticks(scale_rows['scale_B'].tolist())
    ax.set_xticklabels([scale_xlabel(r['model'], r['scale_B'])
                        for _,r in scale_rows.iterrows()], fontsize=9)
    ax.set_xlabel('LLM Scale (approximate parameters)', fontsize=11)
    ax.set_ylabel('Mean Jaccard Similarity', fontsize=11)
    ax.set_title(
        f'Pipeline–LLM Entity Agreement — {cond_label}\n'
        f'(8000 char limit · 183 articles · shading = ±1 SD)', fontsize=12)
    ax.legend(title='Pipeline', fontsize=10)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    ax.grid(axis='y', alpha=0.3); ax.grid(axis='x', alpha=0.15)
    plt.tight_layout()
    for ext in ['pdf','png']:
        fig.savefig(FIGURES_DIR/f'nb13b_scaling_{cond}.{ext}',
                    dpi=300 if ext=='pdf' else 150, bbox_inches='tight')
    plt.show()
    print(f"✓ Figure saved: nb13b_scaling_{cond}.pdf/.png")

# ── Figure 3 : EN vs DE delta (main finding) ──────────────────────────────────
# For each pipeline × model, compute DE-EN difference
delta_records = []
for model_id in model_order:
    for pname in pipeline_lookups.keys():
        en_row = jdf[(jdf['model']==model_id)&(jdf['condition']=='en')&(jdf['pipeline']==pname)]
        de_row = jdf[(jdf['model']==model_id)&(jdf['condition']=='de')&(jdf['pipeline']==pname)]
        if en_row.empty or de_row.empty: continue
        delta_records.append({
            'model'    : model_id,
            'pipeline' : pname,
            'scale_B'  : model_meta[model_id]['scale_B'],
            'en_mean'  : float(en_row['mean'].values[0]),
            'de_mean'  : float(de_row['mean'].values[0]),
            'delta'    : float(de_row['mean'].values[0]) - float(en_row['mean'].values[0]),
        })
delta_df = pd.DataFrame(delta_records)

if not delta_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
    models_sorted = sorted(delta_df['model'].unique(),
                           key=lambda m: model_meta[m]['scale_B'])

    for ax, pname in zip(axes, ['spaCy','Stanza','Flair']):
        psub = delta_df[delta_df['pipeline']==pname]
        if psub.empty: continue
        psub = psub.set_index('model').reindex(models_sorted).reset_index()
        labels  = [scale_xlabel(r['model'], r['scale_B']) for _,r in psub.iterrows()]
        x       = np.arange(len(labels))
        colors  = ['#2ca02c' if d > 0 else '#d62728' for d in psub['delta']]
        bars    = ax.bar(x, psub['delta'], color=colors, alpha=0.8, edgecolor='black', lw=0.5)
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
        ax.set_title(f'{pname}', fontsize=11)
        ax.set_xlabel('Model', fontsize=9)
        if ax == axes[0]: ax.set_ylabel('Jaccard Delta (DE − EN)', fontsize=10)
        for bar, val in zip(bars, psub['delta']):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height() + (0.003 if val >= 0 else -0.008),
                    f'{val:+.3f}', ha='center', va='bottom', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Jaccard Improvement from German Input (DE − EN)\n'
                 '(positive = DE input agrees more with pipeline gold)',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    for ext in ['pdf','png']:
        fig.savefig(FIGURES_DIR/f'nb13b_delta_de_en.{ext}',
                    dpi=300 if ext=='pdf' else 150, bbox_inches='tight')
    plt.show()
    print("✓ Figure saved: nb13b_delta_de_en.pdf/.png")
    print("\nDelta summary (Flair pipeline):")
    flair_delta = delta_df[delta_df['pipeline']=='Flair'][['model','scale_B','en_mean','de_mean','delta']]
    flair_delta = flair_delta.sort_values('scale_B')
    flair_delta[['en_mean','de_mean','delta']] = flair_delta[['en_mean','de_mean','delta']].round(3)
    print(flair_delta.to_string(index=False))


In [ ]:
# ── CELL 12 : SAVE SUMMARY JSON ──────────────────────────────────────────────

run_status = {
    f"{mid}_{cond}": {
        'n_ok'       : len([r for r in ckpt['results'] if r['status']=='ok']),
        'n_failed'   : len(ckpt['failed_ids']),
        'stop_reason': ckpt.get('stop_reason'),
        'text_limit' : TEXT_LIMIT,
    }
    for (mid, cond), ckpt in all_results.items()
}

# Main finding: DE-EN delta for Flair pipeline
flair_delta_dict = {}
if not delta_df.empty:
    for _, row in delta_df[delta_df['pipeline']=='Flair'].iterrows():
        flair_delta_dict[row['model']] = {
            'en_jaccard' : round(row['en_mean'], 4),
            'de_jaccard' : round(row['de_mean'], 4),
            'delta'      : round(row['delta'], 4),
        }

summary = {
    'notebook'       : '13b_en_de_comparison',
    'generated_at'   : datetime.now().isoformat(),
    'n_articles'     : len(ARTICLE_IDS_183),
    'text_limit'     : TEXT_LIMIT,
    'conditions'     : {
        'en': 'English translation input — entity output in English',
        'de': 'German original input — entity output in German (no translation)',
    },
    'models'         : {mid: meta for mid, meta in model_meta.items()},
    'jaccard_full'   : {
        f"{r['pipeline']}|{r['model']}|{r['condition']}": round(r['mean'],4)
        for _,r in jdf.iterrows()
    },
    'de_en_delta_flair': flair_delta_dict,
    'run_status'     : run_status,
    'notes'          : (
        'All runs at 8000 char text limit (96.6% entity coverage by position). '
        'DE prompt includes explicit "do not translate" instruction. '
        'Qwen 7B uses HuggingFace Inference API; skipped if all providers exhausted. '
        'Pipelines run on full article — comparison is conservative lower bound.'
    ),
}

with open(DATA_PROC / 'nb13b_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("="*64)
print("NOTEBOOK 13b COMPLETE")
print("="*64)
print("  nb13b_jaccard_results.csv   — full Jaccard matrix")
print("  nb13b_summary.json          — summary + DE-EN delta")
print("  nb13b_scaling_en.pdf/.png   — scaling curve (EN input)")
print("  nb13b_scaling_de.pdf/.png   — scaling curve (DE input)")
print("  nb13b_delta_de_en.pdf/.png  — EN→DE improvement per model")
print()
print("DE-EN Jaccard delta (Flair pipeline — main finding):")
for model, vals in flair_delta_dict.items():
    scale = model_meta.get(model,{}).get('scale_B','?')
    print(f"  {model:<40} {scale}B  "
          f"EN={vals['en_jaccard']:.3f}  DE={vals['de_jaccard']:.3f}  "
          f"delta={vals['delta']:+.3f}")
